# RSNA Knee — LLM labels (Qwen2.5-Instruct, local, 4bit)
レポートをKaggleのGPU上のオープンな重みのLLMで読み、12所見のラベル(確率つき)を作る。
まず正解ラベル付き58件で規則と比較する。`RUN_ALL=True` で全レポートを処理して `labels_out/llm_labels.csv` を書く。レポート本文は外に出さない。

In [ ]:
import os, sys, subprocess, warnings
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # メモリの断片化によるOOMを減らす
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
REPO = Path('/kaggle/temp/repo') if Path('/kaggle/temp').exists() else Path('/tmp/repo')   # /kaggle/working に置くと実行の出力に含まれてしまう
SRC = REPO/'rsna-knee-abnormality-detection'/'src'
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb', str(REPO)], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS
from multilingual_labels import add_pseudo_labels_ml
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
os.environ['HF_HOME'] = '/kaggle/temp/hf' if Path('/kaggle/temp').exists() else '/tmp/hf'   # モデルのキャッシュは出力に含めない
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes', 'accelerate'], check=False)
import time, torch, transformers
import llm_labels as LL
print('transformers', transformers.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
from multilingual_labels import label_report_lang
from sklearn.metrics import roc_auc_score
train = pd.read_csv(COMP/'train.csv')
train['has_true'] = train['ACL'].notna()
train = add_pseudo_labels_ml(train)
tv = train[train.has_true].reset_index(drop=True)
T = tv[LABELS].values.astype(int)
R = np.array([[label_report_lang(r, l)[c] for c in LABELS] for r, l in zip(tv.Report.fillna(''), tv.lang)])   # 規則ラベル(正解を見ずに作る)
print('true-labeled:', len(tv), tv.lang.value_counts().to_dict())

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
def load(name, four_bit):
    tok = AutoTokenizer.from_pretrained(name)
    kw = dict(device_map='auto')
    if four_bit:
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    else:
        kw['torch_dtype'] = torch.float16
    return tok, AutoModelForCausalLM.from_pretrained(name, **kw).eval()
MODEL_NAME, BS = 'Qwen/Qwen2.5-7B-Instruct', 8
try:
    tok, model = load(MODEL_NAME, four_bit=True)
except Exception as e:                       # bitsandbytesが使えない等: 3Bをfp16で
    print('7B 4bit failed ->', repr(e)[:200]); MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
    tok, model = load(MODEL_NAME, four_bit=False)
print('model', MODEL_NAME, '| GPU memory GB', round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
t0 = time.time()
H, P, npar = LL.label_reports(model, tok, tv.Report.fillna('').tolist(), bs=BS)
sec = (time.time() - t0) / len(tv)
print(f'{sec:.2f} s/report -> all {len(train)} reports ~ {sec * len(train) / 60:.0f} min')
print('reports where fewer than 12 keys were parsed:', int((npar < 12).sum()), '| soft labels from scores:', int((~np.isnan(P)).all(1).sum()))

In [ ]:
def pr(pred):
    tp, fp, fn = (pred & T).sum(0), (pred & (1 - T)).sum(0), ((1 - pred) & T).sum(0)
    return tp / np.maximum(tp + fp, 1), tp / np.maximum(tp + fn, 1)
rp, rr = pr(R); lp, lr = pr(H.astype(int))
def auc(S):
    return np.array([roc_auc_score(T[:, j], S[:, j]) if len(np.unique(T[:, j])) > 1 else np.nan for j in range(len(LABELS))])
B = 0.5 * P + 0.5 * R                                  # LLM確率と規則の平均
tab = pd.DataFrame({'n_pos': T.sum(0), 'rule_P': rp, 'rule_R': rr, 'llm_P': lp, 'llm_R': lr,
                    'AUC_rule': auc(R), 'AUC_llm': auc(P), 'AUC_blend': auc(B)}, index=LABELS).round(2)
print(tab)
print('macro AUC vs TRUE labels  rule %.3f | llm %.3f | blend %.3f' % (np.nanmean(auc(R)), np.nanmean(auc(P)), np.nanmean(auc(B))))
for grp, m in (('en', (tv.lang == 'en').values), ('non-en', (tv.lang != 'en').values)):
    if m.sum():
        f = lambda X: ((X[m] & T[m]).sum() / max(X[m].sum(), 1), (X[m] & T[m]).sum() / max(T[m].sum(), 1))
        print(grp, 'n=%d  rule P/R %.2f/%.2f  llm P/R %.2f/%.2f' % ((m.sum(),) + f(R) + f(H.astype(int))))

In [ ]:
RUN_ALL = True       # True: 全レポートを処理して labels_out/llm_labels.csv に書く(評価表は先に出る)(途中経過も毎チャンク保存)
if RUN_ALL:
    out = Path('labels_out'); out.mkdir(exist_ok=True)
    reports = train.Report.fillna('').tolist(); parts = []
    for s in range(0, len(reports), 400):
        h, p, n = LL.label_reports(model, tok, reports[s:s + 400], bs=BS, log_every=20)
        parts.append(pd.DataFrame(p, columns=LABELS).assign(parsed=n))
        df = pd.concat(parts, ignore_index=True)
        df.insert(0, 'StudyInstanceUID', train.StudyInstanceUID.values[:len(df)])
        df.to_csv(out / 'llm_labels.csv', index=False); print('saved', len(df), '/', len(reports))